# LAGN XGBoost Classifier

*Notebook written by Zach Gillis (zgillis@stanford.edu)*

*Date last updated: March 20, 2026*

This is the last of three notebooks to prepare the *First Cut* filter to select lensed-AGN candidates from the Rubin Observatory alert stream using difference image analysis sources (DIASources).

The previous notebook (`2_inj_source_characterization.ipynb`) built the labeled training dataset by cross-matching Shenming's injected LAGN DIASources against injection coordinates and combining them with non-LAGN DP1 DIASources.

This notebook trains an XGBoost binary classifier to distinguish LAGN DIASources from non-LAGN DIASources, evaluates it on a held-out test set, prepares a completeness/purity curve using estimated LSST survey statistics, and compares the ROC curves of the XGBoost binary classifier with the simple cuts filter. 

## Overview

1. **Load & split data** — load `combined_training_data.csv`, select features, and split into stratified train/test sets
2. **Hyperparameter search** — 5-fold stratified CV with `GridSearchCV` over key XGBoost parameters, scored by ROC-AUC, which is invariant to class ratio
3. **Train final model** — fit XGBoost with best parameters
4. **Evaluate** — classification metrics and diagnostic plots on the held-out test set
5. **Survey scale completeness/purity analysis** — rescale classifier performance to the true LSST class ratio (~1:37,000 LAGN:non-LAGN)
6. **Simple cuts comparison** — compare XGBoost classifier against the simple cuts filter by sweeping the `moment_ext` threshold
7. **Validation on uninjected data** — apply the classifier to DIASources from visits with no injected sources to assess effects of potential Rubin data processing pipeline differences

---
## Section 1 — Setup & Data Loading

### Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import yaml
import corner

from matplotlib.patches import Patch
from matplotlib.lines import Line2D

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV, cross_val_score
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    roc_curve, auc, confusion_matrix,
    precision_recall_curve, average_precision_score, matthews_corrcoef,
)
from sklearn.calibration import calibration_curve

### Load & Split Data

Loads `combined_training_data.csv` (produced in the previous notebook) — which contains one row per DIASource with `label=0` (non-LAGN DIASource, DP1) or `label=1` (LAGN DIASource, injected).

A subset of features is selected from the candidate features in `combined_training_data.csv`. Currently, several features are commented out to reduce overfitting risk (`centroid_flag`, `trailLength`, `trailFlux`, and the dipole error/flux columns). The long-term solution to properly assess overfitting is a truly independent test set drawn from a separate set of injected lenses not seen during training (see below).

The dataset is split 80% train / 20% test with `stratify=y` to preserve the class ratio in both splits.

#### Band Weighting

The injected LAGN (label=1) and DP1 background (label=0) have different band distributions. Without correction, the model could implicitly learn band membership as a proxy for class label rather than the underlying physical features.

To address this, each training sample is assigned a weight inversely proportional to the count of its `(label, band)` cell, normalized so the mean weight is 1. 

This ensures every `(label, band)` combination contributes equally to the loss. Because LAGN cells are ~400–800× smaller than non-LAGN cells, these weights also subsume the overall class imbalance correction. Weights are passed to `model.fit()` via `sample_weight` and recomputed per fold in the Section 5 cross-validation.

#### Limitations of the 80/20 Split

The current train/test split is a random 80/20 partition of the same set of lenses. Because multiple DIASources from the same injected lens appear in both train and test, the model has effectively seen each lens's morphology during training. This inflates test-set performance metrics and means the evaluation does not truly measure generalization to new lenses. We'll eventually need to hold out a separate set of injected lenses entirely, so no DIASources from those lenses appear in training. This requires many more lenses. 

XGBoost natively supports NaN values (present in dipole-specific columns in DIASources without attempted dipole fits) and categorical variables (i.e. `band`).

**Active features:** `band`, `template_flux`, `scienceFlux`, `psfFlux`, `apFlux`, `temp_sci_flux_ratio`, `moment_ext`, `ellip_ext`, `flux_ext`, `extendedness`, `psfChi2`, `isDipole`, `dipoleFitAttempted`, `dipoleChi2`, `dipoleLength`, `x_y_err`

In [ ]:
FEATURES = [
    # Other
    'band',
    # 'centroid_flag',
    'psf_fwhm',
    'snr',
    
    # Flux
    'template_flux',
    'scienceFlux',
    'psfFlux',
    'apFlux',
    'temp_sci_flux_ratio',
    
    # Extended
    'moment_ext',
    'ellip_ext',
    'flux_ext',
    'extendedness',
    'psfChi2',
    # 'trailFlux',
    # 'trailLength',
    
    # Dipole
    # 'isDipole',
    'dipoleFitAttempted',
    'dipoleChi2',
    # 'dipoleFluxDiffErr',
    # 'dipoleMeanFlux',
    # 'dipoleMeanFluxErr',
    'dipoleLength',
    
    # Centroid
    'x_y_err',
]

df = pd.read_csv('combined_training_data.csv')

df['isDipole'] = df['isDipole'] == 'True'
df['band'] = pd.Categorical(df['band'])
band_categories = df['band'].cat.categories

X       = df.drop(columns=['label'])[FEATURES]
y       = df['label']
lens_id = df['lens_id']  # injection_coords index for LAGN sources; -1 for non-LAGN

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples     : {len(X_train):,}")
print(f"Test samples         : {len(X_test):,}")

# Per-(label, band) sample weights: each (class, band) cell contributes equally.
# Prevents the model from learning band as a proxy for class.
def compute_band_weights(y_s, X_s):
    tmp = pd.DataFrame({'label': y_s.values, 'band': X_s['band'].values})
    cell_counts = tmp.groupby(['label', 'band'], observed=True)['label'].transform('count')
    w = 1.0 / cell_counts
    return (w / w.mean()).values  # normalize so mean weight = 1

sample_weight_train = compute_band_weights(y_train, X_train)

print("\nTraining samples by (label, band):")
print(pd.crosstab(y_train, X_train['band']))

tmp = pd.DataFrame({'label': y_train.values, 'band': X_train['band'].values, 'weight': sample_weight_train})
print("\nMean sample weight by (label, band):")
print(tmp.groupby(['label', 'band'], observed=True)['weight'].mean().unstack(fill_value=0).round(3))

---
## Section 2 — Hyperparameter Search

### Grid Search with 5-Fold Cross-Validation

Searches over a grid of XGBoost hyperparameters using `GridSearchCV` with 5-fold stratified cross-validation, scored by ROC-AUC.

ROC-AUC is used because it is invariant to class ratio — this data has a about 1 lens : 50 non-lens ratio compared to the expected survey 1 lens : 37,000 non-lens ratio.

An improvement to be made is to create folds based on `lens_id`, so that each of the folds uses a different set of lenses. To use `lens_id` would require lens injections to be unique. If not, using an additional validation set (with its own unique lenses) for hyperparameter tuning may be necessary. 

There are CV options other than grid search; this is the simplest but takes the most amount of time. 

You can also skip this step and manually set parameters in the next code block. 

In [ ]:
param_grid = {
    'max_depth':        [4, 6, 8],
    'learning_rate':    [0.01, 0.02, 0.03],
    'subsample':        [0.6, 0.8],
    'colsample_bytree': [0.7, 0.9],
}

search = GridSearchCV(
    XGBClassifier(
        n_estimators=500,
        scale_pos_weight=scale_pos_weight,
        eval_metric='logloss',
        enable_categorical=True,
        random_state=42,
        n_jobs=-1,
    ),
    param_grid=param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc',
    n_jobs=-1,
    refit=False,
    verbose=1,
)
search.fit(X_train, y_train, sample_weight=sample_weight_train)

best_params = search.best_params_
print(f"\nBest params: {best_params}")

---
## Section 3 — Train Final Model

### Fit XGBoost with Early Stopping

Trains the final classifier using the best hyperparameters from the grid search (or set `USE_BEST_PARAMS = False` if you'd like to set your own parameters manually). The manual parameters here are the best hyperparameters from a previous run. 

Right now, early stopping doesn't do anything, indicating that the test set is not preventing overfitting. Updated test-train split required.

In [ ]:
# ── Train Final Model ──────────────────────────────────────────────────────────
# Set USE_BEST_PARAMS = False to train with the default hyperparameters instead.
USE_BEST_PARAMS = False

MANUAL_PARAMS = {
    'max_depth':        6,
    'learning_rate':    0.03,
    'subsample':        0.6,
    'colsample_bytree': 0.7,
}

params = best_params if USE_BEST_PARAMS else MANUAL_PARAMS

model = XGBClassifier(
    n_estimators=500,
    eval_metric='logloss',
    enable_categorical=True,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50,
    **params,
)
model.fit(X_train, y_train, sample_weight=sample_weight_train,
          eval_set=[(X_test, y_test)], verbose=False)
print(f"Training complete. Best n_estimators: {model.best_iteration}")

---
## Section 4 — Evaluation on Held-Out Test Set

### Test Set Metrics

Evaluates the trained model on `X_test` and `y_test`, the held-out test set from earlier. The precision/recall is skewed by the large class imbalance. 

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Non-LAGN (0)', 'LAGN (1)'], digits=4))

### Visualization: Diagnostic Plots

Five diagnostic plots summarizing classifier performance on the held-out test set:

| Plot | Description |
|---|---|
| **Feature Importance** | XGBoost gain-based importance for each input feature |
| **ROC Curve** | True positive rate vs. false positive rate across all thresholds |
| **Precision-Recall Curve** | Precision vs. recall at the training-set class ratio (~1:50) |
| **P(LAGN) Distribution** | Predicted probability histograms for LAGN and non-LAGN DIASources |
| **Confusion Matrix** | Counts and row-normalized rates at nominal threshold (0.5) |

In [ ]:
fig = plt.figure(figsize=(10, 6), dpi=300)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.35)

# Feature Importance
ax1 = fig.add_subplot(gs[0, 0])
importance = model.feature_importances_
feat_names = np.array(X.columns.tolist())
top_idx = np.argsort(importance)[-20:]
ax1.barh(feat_names[top_idx], importance[top_idx], color='steelblue', edgecolor='white')
ax1.set_xlabel('Gain', fontsize=11)
ax1.set_title('Feature Importance', fontsize=12, fontweight='bold')
ax1.tick_params(axis='y', labelsize=8)
ax1.spines[['top', 'right']].set_visible(False)

# ROC Curve
ax2 = fig.add_subplot(gs[0, 1])
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc_val = auc(fpr, tpr)
ax2.plot(fpr, tpr, color='darkorange', lw=2, label=f'XGBoost\n(AUC = {roc_auc_val:.4f})')
ax2.plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--', label='Random')
ax2.fill_between(fpr, tpr, alpha=0.08, color='darkorange')
ax2.set_xlim([0.0, 1.0])
ax2.set_ylim([0.0, 1.02])
ax2.set_xlabel('False Positive Rate', fontsize=11)
ax2.set_ylabel('True Positive Rate', fontsize=11)
ax2.set_title('ROC Curve', fontsize=12, fontweight='bold')
ax2.legend(loc='lower right', fontsize=10)
ax2.spines[['top', 'right']].set_visible(False)

# Precision-Recall Curve
ax3 = fig.add_subplot(gs[0, 2])
pr_precision, pr_recall, _ = precision_recall_curve(y_test, y_prob)
ap = average_precision_score(y_test, y_prob)
baseline = y_test.mean()
ax3.plot(pr_recall, pr_precision, color='darkorange', lw=2, label=f'XGBoost\n(AP = {ap:.4f})')
ax3.axhline(baseline, color='navy', lw=1.5, linestyle='--',
            label=f'Random\n(AP = {baseline:.4f})')
ax3.fill_between(pr_recall, pr_precision, alpha=0.08, color='darkorange')
ax3.set_xlim([0.0, 1.0])
ax3.set_ylim([0.0, 1.02])
ax3.set_xlabel('Recall', fontsize=11)
ax3.set_ylabel('Precision', fontsize=11)
ax3.set_title('Precision-Recall Curve', fontsize=12, fontweight='bold')
ax3.legend(loc='upper right', fontsize=10)
ax3.spines[['top', 'right']].set_visible(False)

# Predicted Probability Distribution
ax4 = fig.add_subplot(gs[1, 0])
bins = np.linspace(0, 1, 60)
ax4.hist(y_prob[y_test == 0], bins=bins, alpha=0.65, color='steelblue',
         label='Non-LAGN (0)', density=True)
ax4.hist(y_prob[y_test == 1], bins=bins, alpha=0.65, color='darkorange',
         label='LAGN (1)', density=True)
ax4.axvline(0.5, color='red', linestyle='--', lw=1.5, label='Threshold = 0.5')
ax4.set_xlabel('P(LAGN)', fontsize=11)
ax4.set_ylabel('Density', fontsize=11)
ax4.set_title('P(LAGN)', fontsize=12, fontweight='bold')
ax4.legend(fontsize=10, loc='upper center')
ax4.spines[['top', 'right']].set_visible(False)

# Confusion Matrix Heatmap
ax5 = fig.add_subplot(gs[1, 1])
cm = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
labels_raw = [f'{v:,}' for v in cm.ravel()]
labels_pct = [f'{v:.1%}' for v in cm_norm.ravel()]
annot = np.array([f'{r}\n({p})' for r, p in zip(labels_raw, labels_pct)]).reshape(2, 2)

sns.heatmap(cm_norm, annot=annot, fmt='', cmap='Blues', ax=ax5,
            xticklabels=['Non-LAGN (0)', 'LAGN (1)'],
            yticklabels=['Non-LAGN (0)', 'LAGN (1)'],
            vmin=0, vmax=1, linewidths=0.5, cbar_kws={'label': 'Row-normalised rate'})
ax5.set_xlabel('Predicted Label', fontsize=11)
ax5.set_ylabel('True Label', fontsize=11)
ax5.set_title('Confusion Matrix', fontsize=12, fontweight='bold')

plt.show()

# Print feature importance
sorted_idx = np.argsort(importance)[::-1]
print('Full feature importance:')
for rank, i in enumerate(sorted_idx, 1):
    print(f"{rank:>3}. {feat_names[i]:<35} {importance[i]:.6f}")                                      
                         

---
## Section 5 — Survey Scale Analysis

The test set metrics above reflect a ~1:50 class ratio, not the LSST DIASource LAGN to non-LAGN ratio of ~1:37,000. At survey scale, the same classifier will produce far more false positives relative to true positives.

Here we use only invariant quantities like true positive rate (completeness, TPR) and false positive rate (FPR) to prepare a representative completeness/purity curve.

## Survey Population Estimates

We estimate the number of LAGN and non-LAGN DIASources expected in the 10-year LSST survey and in year 1. These are rough calculations.

### LAGN DIASources

| Parameter | Value |
|---|---|
| Expected lensed AGN in full survey | 4,000 |
| Visits per sky position (over 10 years) | 800 |
| LAGN detection rate (DIASource per visit per LAGN) | 10% |
| **Total LAGN DIASources (10 yr)** | **320,000** |
| **Total LAGN DIASources (yr 1, 80 visits)** | **32,000** |

### Non-LAGN DIASources

Non-LAGN DIASources are estimated from the LSST DP1 ECDFS (within the wide fast deep survey area).

| Parameter | Value |
|---|---|
| DP1 DIASources (ECDFS) | 551,975 |
| DP1 visits | 855 |
| DP1 field area | 0.785 deg² |
| DIASources / visit / deg² | 822.4 / visit / deg² |
| Full survey area | 18,000 deg² |
| **Total background DIASources (10 yr)** | **~11.8 billion** |
| **Total background DIASources (yr 1)** | **~1.18 billion** |


The ratio of non-LAGN to LAGN DIASources is **~37,000:1**.

### Completeness–Purity Analysis

We sweep the classification threshold and compute the completeness and purity at survey scale using 10-fold stratified cross-validation (for smoothness).

Completeness (TPR) and FPR are can be measured on the training data and then rescaled to yield counts of true positives and false positives:

$$N_\text{LAGN,yr1} = 32,000$$
$$N_\text{non-LAGN,yr1} = 1,184,257,459$$

$$\text{TP} = \text{TPR} \times N_\text{LAGN,yr1}$$
$$\text{FP} = \text{FPR} \times N_\text{non-LAGN,yr1}$$
$$\text{purity} = \frac{\text{TP}}{\text{TP} + \text{FP}}$$

The purity is averaged across all 10 folds. Two plots are produced:
1. **Completeness vs. Purity**
2. **LAGN DIASources vs. Purity** (how many LAGN DIASources detected at each purity level in year 1)

The commented out plotting block—unique LAGN recovered—uses `lens_id` (lenses are not unique in this iteration). Future completeness-purity analysis will likely be better considering LAGN rather than DIASources; therefore, using `lens_id`. 

In [ ]:
lagn_diasources     = 32_000
non_lagn_diasources = 1_184_257_459
N_lenses_yr1        = 4_000

print(f"LAGN DIASources     (yr 1) : {lagn_diasources:,}")
print(f"Non-LAGN DIASources (yr 1) : {non_lagn_diasources:,}")
print(f"Non-LAGN:LAGN ratio        : 1:{non_lagn_diasources // lagn_diasources:,}")

cv          = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
comp_grid   = np.linspace(0, 1, 100)
fold_purity     = []
fold_lens_frac  = []

for train_idx, val_idx in cv.split(X, y):
    fold_sw = compute_band_weights(y.iloc[train_idx], X.iloc[train_idx])

    fold_model = XGBClassifier(
        n_estimators=300,
        eval_metric='logloss', enable_categorical=True,
        random_state=42, n_jobs=-1, **params,
    )
    fold_model.fit(X.iloc[train_idx], y.iloc[train_idx],
                   sample_weight=fold_sw, verbose=False)

    # Get predicted probabilities and true labels for validation fold
    prob   = fold_model.predict_proba(X.iloc[val_idx])[:, 1]
    labels = np.array(y.iloc[val_idx])

    # Sort validation DIASources by descending predicted probability
    order  = np.argsort(prob)[::-1]
    labels = labels[order]

    # Total positives and negatives in this fold
    N_pos = (labels == 1).sum()
    N_neg = (labels == 0).sum()

    # Cumulative TP, FP, TN, FN as we descend the sorted validation DIASources
    TP = np.cumsum(labels == 1)
    FP = np.cumsum(labels == 0)

    # Calculate TPR and FPR as we descend the sorted validation DIASources
    tpr = TP / N_pos
    fpr = FP / N_neg

    # Scale to survey population to get true purity
    TP_survey = tpr * lagn_diasources
    FP_survey = fpr * non_lagn_diasources
    purity    = TP_survey / (TP_survey + FP_survey).clip(1e-12)

    fold_purity.append(np.interp(comp_grid, tpr, purity))

    # Lens recovery: count unique lenses recovered as we descend the sorted val set.
    # A lens is "recovered" the first time any of its DIASources appears in the selection.
    val_lens_id      = np.array(lens_id.iloc[val_idx])[order]
    N_lenses_fold    = len(np.unique(val_lens_id[labels == 1]))
    seen_lenses      = set()
    n_lenses_cumul   = np.zeros(len(labels), dtype=int)
    for j in range(len(labels)):
        if labels[j] == 1:
            seen_lenses.add(val_lens_id[j])
        n_lenses_cumul[j] = len(seen_lenses)
    lens_frac = n_lenses_cumul / max(N_lenses_fold, 1)

    fold_lens_frac.append(np.interp(comp_grid, tpr, lens_frac))

mean_purity      = np.mean(fold_purity, axis=0)
mean_lens_frac   = np.mean(fold_lens_frac, axis=0)
sample_size_yr1  = comp_grid * lagn_diasources
lenses_yr1       = mean_lens_frac * N_lenses_yr1

# Threshold at target completeness levels (using full model on test set)
fpr_curve, tpr_curve, thresholds = roc_curve(y_test, y_prob)
for target in [0.30, 0.90]:
    thresh = np.interp(target, tpr_curve, thresholds)
    print(f"Threshold at {target:.0%} completeness: {thresh:.4f}")

In [ ]:
# Plot 1: Completeness vs. Purity
fig, ax = plt.subplots(figsize=(4, 4), dpi=300)
ax.plot(comp_grid, mean_purity, color='steelblue', lw=2)
ax.set_yscale('log')
ax.set_ylim(1e-4, 1)
ax.set_xlim(0, 1)
ax.set(xlabel='Completeness', ylabel='Purity',
       title='Completeness–Purity (10-fold CV)')
ax.grid(True, which='both', ls='--', lw=0.5, alpha=0.6)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

# Plot 2+3+4: DIASources, Unique LAGN, and Total Alerts vs. Purity
total_alerts_yr1 = sample_size_yr1 / np.clip(mean_purity, 1e-12, None)

fig, ax1 = plt.subplots(figsize=(6, 4), dpi=300)
fig.subplots_adjust(right=0.75)

ax1.plot(mean_purity, sample_size_yr1, color='darkorange', lw=2, label='DIASources from LAGN')
ax1.set_xscale('log')
ax1.set_xlim(1e-4, 1)
ax1.set_xlabel('Purity')
ax1.set_ylabel('DIASources from LAGN (year 1)', color='darkorange')
ax1.tick_params(axis='y', labelcolor='darkorange')
ax1.spines[['top']].set_visible(False)
ax1.set_title('Sample Composition vs. Purity (year 1)')
ax1.grid(True, which='both', ls='--', lw=0.5, alpha=0.6)

# this I removed for now until we get unique lenses. I'm also not sure the scaling makes sense. 
# ax2 = ax1.twinx()
# ax2.plot(mean_purity, lenses_yr1, color='forestgreen', lw=2, label='Unique LAGN')
# ax2.set_ylabel('Unique LAGN recovered (year 1)', color='forestgreen')
# ax2.tick_params(axis='y', labelcolor='forestgreen')
# ax2.spines[['top']].set_visible(False)

ax3 = ax1.twinx()
ax3.spines['right'].set_position(('outward', 0))
ax3.plot(mean_purity, total_alerts_yr1, color='slategrey', lw=2, label='Total alerts')
ax3.set_ylabel('Total alerts in sample (year 1)', color='slategrey')
ax3.set_yscale('log')
ax3.tick_params(axis='y', labelcolor='slategrey')
ax3.spines[['top']].set_visible(False)

# lines = [ax1.get_lines()[0], ax2.get_lines()[0], ax3.get_lines()[0]]
lines = [ax1.get_lines()[0], ax3.get_lines()[0]]
ax1.legend(lines, [l.get_label() for l in lines], loc='lower left', fontsize=9)

plt.show()

### XGBoost vs. Simple Cuts Filter

Compares ROC curves for XGBoost and the simple cuts filter. Three curves are shown:

- **XGBoost binary classifier**
- **All cuts, sweep `flux_ext`** — holds `ellip_ext` and `temp_sci_flux_ratio` at their fixed thresholds and sweeps `flux_ext`. The line doesn't reach a completeness of one, as the other cuts remove LAGN DIASources before we even sweep `flux_ext`
- **`flux_ext` only** — sweeps `flux_ext` alone with no other cuts applied

Note: I switched from `moment_ext` to `flux_ext` for the sweep due to the sizable number of NaN moment measurements. 

In [ ]:
with open('simple_cuts_thresholds.yaml') as f:
    sc = yaml.safe_load(f)

# XGBoost ROC
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_prob)
auc_xgb = roc_auc_score(y_test, y_prob)

# flux_ext only ROC
valid = X_test['flux_ext'].notna()
fpr_iext, tpr_iext, _ = roc_curve(y_test[valid], X_test.loc[valid, 'flux_ext'])
auc_iext = roc_auc_score(y_test[valid], X_test.loc[valid, 'flux_ext'])

# Full simple cuts ROC, sweeping flux_ext
flux_cap_array = np.array([sc['flux_caps'][band] for band in X_test['band']])
passes_other = (
    (X_test['ellip_ext']           > sc['ellip_ext'])          &
    (X_test['temp_sci_flux_ratio'] > sc['temp_sci_flux_ratio']) &
    (X_test['template_flux']       < flux_cap_array)           &
    X_test['flux_ext'].notna()
)

flux_ext_sub = X_test.loc[passes_other, 'flux_ext'].values
y_sub        = y_test[passes_other].values
N_pos_full   = (y_test == 1).sum()
N_neg_full   = (y_test == 0).sum()

order      = np.argsort(flux_ext_sub)[::-1]
labels_ord = y_sub[order]
TP = np.cumsum(labels_ord == 1)
FP = np.cumsum(labels_ord == 0)
tpr_sc = np.concatenate([[0], TP / N_pos_full])
fpr_sc = np.concatenate([[0], FP / N_neg_full])

# Plot
fig, ax = plt.subplots(figsize=(5, 4), dpi=300)

ax.plot(fpr_xgb,  tpr_xgb,  color='darkorange', lw=2, label=f'XGBoost (AUC = {auc_xgb:.4f})')
ax.plot(fpr_iext, tpr_iext, color='steelblue',  lw=2, label=f'flux_ext only (AUC = {auc_iext:.4f})')
ax.plot(fpr_sc,   tpr_sc,   color='steelblue',  lw=2, ls='--', label='All cuts, sweep flux_ext')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.grid(alpha=0.3)
plt.show()


### Visualization: Corner Plot of XGBoost Features

Visualizes the feature space used in XGBoost training (excluding band), overlaying three populations:
- **Grey** — Non-LAGN (DP1) (label = 0)
- **Green** — LAGN (injected) (label = 1)
- **Red** — XGBoost candidates (threshold = 0.5)

Features with log-scale axes are sanitized to remove non-positive values before passing to `corner`. Columns where any population has fewer than 2 unique in-range values are excluded. 

In [ ]:
FEATURE_PROPS = {
    'psf_fwhm':            ('linear', (0, 10),        'PSF FWHM'),
    'snr':                 ('linear', (0, 50),         'SNR'),
    'template_flux':       ('log',    (1e2, 1e7),      'Temp. Flux'),
    'scienceFlux':         ('log',    (1e2, 1e7),      'Sci. Flux'),
    'psfFlux':             ('log',    (1e2, 1e7),      'PSF Flux'),
    'apFlux':              ('log',    (1e2, 1e7),      'Ap. Flux'),
    'temp_sci_flux_ratio': ('linear', (0, 2),          'Temp./Sci.\nFlux'),
    'moment_ext':          ('linear', (0, 4),          'Moment Ext.'),
    'ellip_ext':           ('linear', (0, 1),          'Ellip. Diff.'),
    'flux_ext':            ('log',    (0.1, 10.0),     'Flux Ext.'),
    'extendedness':        ('linear', (0, 1),          'Rubin Ext.'),
    'psfChi2':             ('log',    (1, 1e6),        'PSF χ²'),
    'isDipole':            ('linear', (-0.1, 1.1),     'Is Dipole'),
    'dipoleFitAttempted':  ('linear', (-0.1, 1.1),     'Dip. Fit?'),
    'dipoleChi2':          ('log',    (1e-2, 1e4),     'Dip. χ²'),
    'dipoleLength':        ('linear', (0, 0.5),        'Dip. Length'),
    'x_y_err':             ('linear', (0, 4),          'Cent. Err.'),
    'trailLength':         ('linear', (0, 4),          'Trail Length'),
    'trailFlux':           ('log',    (1e2, 1e7),      'Trail Flux'),
    'dipoleMeanFlux':      ('log',    (1e2, 1e7),      'Dipole Mean Flux'),
    'dipoleFluxDiffErr':   ('log',    (1e0, 1e5),      'Dipole Flux Diff Err'),
    'dipoleMeanFluxErr':   ('log',    (1e0, 1e5),      'Dipole Mean Flux Err'),
    'centroid_flag':       ('linear', (-0.1, 1.1),     'Centroid Flag'),
}

def to_array(subset, cols):
    arr = np.empty((len(subset), len(cols)), dtype=float)
    for i, c in enumerate(cols):
        scale, (lo, hi) = FEATURE_PROPS[c][0], FEATURE_PROPS[c][1]
        s = pd.to_numeric(subset[c], errors='coerce').replace([np.inf, -np.inf], np.nan)
        if scale == 'log':
            s = s.where(s > 0)      # mask non-positive: log(≤0) is undefined
        med = s.median()
        fill = med if np.isfinite(med) else lo   # fall back to range lower bound
        s = s.fillna(fill)
        arr[:, i] = s.values
        # Final safety: clamp any surviving bad values
        if scale == 'log':
            bad = ~np.isfinite(arr[:, i]) | (arr[:, i] <= 0)
            arr[bad, i] = lo
        else:
            arr[~np.isfinite(arr[:, i]), i] = 0.0
    return arr

df_false = df[df['label'] == 0]
df_true  = df[df['label'] == 1]

XGB_THRESHOLD = 0.5
all_probs = model.predict_proba(X)[:, 1]
df_xgb = df[all_probs > XGB_THRESHOLD]
print(f"Sources passing XGBoost (threshold={XGB_THRESHOLD}): {len(df_xgb):,}")

candidate_cols = [f for f in FEATURES if f != 'band']

# Drop columns where any population has fewer than 2 unique in-range values
def is_plottable(col, *dfs, min_samples=2):
    scale, (lo, hi) = FEATURE_PROPS[col][0], FEATURE_PROPS[col][1]
    for d in dfs:
        s = pd.to_numeric(d[col], errors='coerce')
        if scale == 'log':
            s = s.where(s > 0)
        vals = s.dropna()
        in_range = vals[(vals >= lo) & (vals <= hi)]
        if in_range.nunique() < min_samples:
            return False
    return True

columns_corner = [c for c in candidate_cols if is_plottable(c, df_false, df_true, df_xgb)]
dropped = set(candidate_cols) - set(columns_corner)
if dropped:
    print(f"Dropped unplottable columns: {dropped}")

axes_scale = [FEATURE_PROPS[f][0] for f in columns_corner]
ranges     = [FEATURE_PROPS[f][1] for f in columns_corner]
labels     = [FEATURE_PROPS[f][2] for f in columns_corner]

data_array_all  = to_array(df_false, columns_corner)
data_array_lagn = to_array(df_true,  columns_corner)
data_array_xgb  = to_array(df_xgb,   columns_corner)

fig = corner.corner(data_array_all,
                    labels=labels,
                    axes_scale=axes_scale,
                    range=ranges,
                    fill_contours=True,
                    smooth=0.7,
                    show_titles=False,
                    color='grey',
                    plot_datapoints=False,
                    plot_contours=True,
                    plot_density=True,
                    bins=20,
                    fig=plt.figure(figsize=(15,15), dpi=300),
                    max_n_ticks=3,
                    hist_kwargs=dict(density=True)
                   )

fig = corner.corner(data_array_lagn,
                    labels=labels,
                    axes_scale=axes_scale,
                    range=ranges,
                    fill_contours=True,
                    smooth=0.7,
                    show_titles=False,
                    color='green',
                    plot_datapoints=False,
                    plot_contours=True,
                    plot_density=True,
                    bins=20,
                    fig=fig,
                    max_n_ticks=3,
                    hist_kwargs=dict(density=True)
                   )

fig = corner.corner(data_array_xgb,
                    labels=labels,
                    axes_scale=axes_scale,
                    range=ranges,
                    fill_contours=True,
                    smooth=0.7,
                    show_titles=False,
                    color='red',
                    plot_datapoints=False,
                    plot_contours=True,
                    plot_density=True,
                    bins=20,
                    fig=fig,
                    max_n_ticks=3,
                    hist_kwargs=dict(density=True)
                   )

for ax in fig.axes:
    ax.tick_params(labelsize=8, axis='both', which='major', pad=2)
    for label in ax.get_xticklabels():
        label.set_rotation(0)
    xlabels = [t for t in ax.xaxis.get_major_ticks() if t.label1.get_visible()]
    if xlabels:
        xlabels[-1].label1.set_visible(False)
    ylabels = [t for t in ax.yaxis.get_major_ticks() if t.label1.get_visible()]
    if ylabels:
        ylabels[0].label1.set_visible(False)

equation_text = (
    r'$\mathrm{Flux\ Ext.} = \dfrac{F_\mathrm{ap}}{F_\mathrm{PSF}}$' + '\n\n' +
    r'$\mathrm{Moment\ Ext.} = \dfrac{I_{xx} + I_{yy}}{I_{xx}^{\,\mathrm{PSF}} + I_{yy}^{\,\mathrm{PSF}}}$' + '\n\n' +
    r'$\mathrm{Ellip.\ Diff.} = \dfrac{\sqrt{(I_{xx}-I_{yy})^2+4I_{xy}^2}}{I_{xx}+I_{yy}} - '
    r'\dfrac{\sqrt{(I_{xx}^{\,\mathrm{PSF}}-I_{yy}^{\,\mathrm{PSF}})^2+4(I_{xy}^{\,\mathrm{PSF}})^2}}'
    r'{I_{xx}^{\,\mathrm{PSF}}+I_{yy}^{\,\mathrm{PSF}}}$'
)

fig.text(0.55, 0.87, equation_text, fontsize=13,
         bbox=dict(boxstyle='round,pad=0.8', facecolor='white', edgecolor='white', alpha=0),
         verticalalignment='top')

legend_elements = [
    Patch(facecolor='grey',  edgecolor='black',    label=f'All DP1 DIASources ({len(df_false):,})'),
    Patch(facecolor='green', edgecolor='darkgreen', label=f'Injected LAGN ({len(df_true):,})'),
    Patch(facecolor='red',   edgecolor='darkred',   label=f'XGBoost p > {XGB_THRESHOLD} ({len(df_xgb):,})'),
]
fig.legend(handles=legend_elements, loc='upper right', fontsize=13, framealpha=0.9)

plt.show()

---
## Section 6 — Validation on Uninjected Data within Injection Pipeline

Applies the trained classifier to DIASources from visits in the injection pipeline where no sources were actually injected.

This serves as a sanity check to see whether the classifier is detecting differences in the data pipelines (DP1 uses an older difference image analysis pipeline).

A significantly elevated FPR on non-LAGN from source injection pipeline compared to non-LAGN from DP1 would indicate the model has learned pipeline-specific artifacts rather than physical features.

In [ ]:
# file created in notebook 2
noinj = pd.read_csv('diasources_no_inj.csv')

noinj['isDipole'] = noinj['isDipole'] == 'True'
noinj['band'] = pd.Categorical(noinj['band'], categories=band_categories)

X_noinj = noinj[X.columns]

noinj_pred = model.predict(X_noinj)
noinj_prob = model.predict_proba(X_noinj)[:, 1]

threshold = 0.5

fpr_test   = (y_pred[y_test == 0] == 1).mean()
fpr_noinj  = noinj_pred.mean()

print(f"FPR on held-out test set : {fpr_test:.4f}  ({fpr_test*100:.2f}%)")
print(f"FPR on uninjected data   : {fpr_noinj:.4f}  ({fpr_noinj*100:.2f}%)")

prob_bg_test = y_prob[y_test == 0]

fig, ax = plt.subplots(figsize=(4, 3), dpi=300)
bins = np.linspace(0, 1, 60)

ax.hist(prob_bg_test, bins=bins, density=True, alpha=0.6,
        color='steelblue', label=f'Non-LAGN (from DP1)\n(FPR = {fpr_test:.3f})')
ax.hist(noinj_prob, bins=bins, density=True, alpha=0.6,
        color='darkorange', label=f'Non-LAGN (from source injection)\n(FPR = {fpr_noinj:.3f})')

ax.set_xlabel('P(LAGN)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()


### (Optional) See DP1 cutouts of false positives

If you'd like to use the ``create_image_gallery()`` function from earlier to see false positives (non-LAGN classified as LAGN), use this code block. Useful for identifying false positive morphologies. 

In [ ]:
import importlib
import data_processing as dp
importlib.reload(dp)
from astropy.table import Table
from astropy.coordinates import SkyCoord
import astropy.units as u

all_probs = model.predict_proba(X)[:, 1]
XGB_THRESHOLD = 0.9047 ## set cutoff

# ── False Positives (DP1 DIASources flagged as LAGN) ──────────────────────────
df_fp = df[(df['label'] == 0) & (all_probs > XGB_THRESHOLD)]
print(f"False positives: {len(df_fp):,}")
fp_table = Table.from_pandas(df_fp)

# Load full DIASource data for all 3 fields
full_table = dp.load_dp1(fetch_from_server=False, load_all_fields=True, service=dp.service)
full_table = dp.add_engineered_features(full_table)

# Match fp_table to full_table by ra/dec (strip any existing units with np.array)
fp_coords   = SkyCoord(ra=np.array(fp_table['ra'])*u.deg,  dec=np.array(fp_table['dec'])*u.deg)
full_coords = SkyCoord(ra=np.array(full_table['ra'])*u.deg, dec=np.array(full_table['dec'])*u.deg)

idx, sep, _ = fp_coords.match_to_catalog_sky(full_coords)
matched_mask = sep < 0.5 * u.arcsec
fp_full = full_table[idx[matched_mask]]
print(f"Matched {matched_mask.sum():,} / {len(fp_table):,} false positives in full table")

fig_lsst, fig_legacy = dp.create_image_gallery(fp_full, rows=3, cols=2, include_legacy=False)
plt.show()